# Insights de Negócio — Don't Go Predictor

**Objetivo deste notebook:** Traduzir os resultados técnicos do modelo preditivo em
linguagem de negócio para os tomadores de decisão.

Tópicos:
1. Comparação com baseline (regra estática) — por que ML supera regras simples
2. Impacto operacional quantificado — o que 22.809 DGs antecipados significa na prática
3. Alarm Fingerprint — quais alarmes precedem Don't Go e por quê isso é acionável
4. Recomendações operacionais concretas

In [ ]:
import sys, json, pickle
sys.path.insert(0, '../src')

import polars as pl
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from pathlib import Path
from sklearn.metrics import f1_score, precision_score, recall_score, confusion_matrix

GOLD_DIR    = Path('../outputs/gold')
SILVER_DIR  = Path('../outputs/silver')
FIGURES_DIR = Path('../outputs/figures')
DASH_DIR    = Path('../outputs/dashboards')

with open(GOLD_DIR / 'lgbm_dontgo.pkl', 'rb') as f:
    model = pickle.load(f)

with open(Path('../outputs/reports/model_metrics.json')) as f:
    final_metrics = json.load(f)

THRESHOLD = final_metrics['optimal_threshold']
print(f"Modelo carregado: {len(model.feature_name_)} features | threshold={THRESHOLD:.4f}")

## 1. LightGBM vs Baseline: Por que ML é necessário?

Um gestor pode perguntar: "não seria mais simples usar uma regra — se houver muitos alarmes
críticos na última hora, emitir alerta?"

Testamos exatamente isso. A regra ótima é: **se `n_criticos_60m ≥ 4`, emitir alerta**.
Threshold selecionado no conjunto de validação (mai/2025).

In [ ]:
# Seleção do threshold do baseline no conjunto de validação (mai/2025)
may_data = pl.read_parquet(str(GOLD_DIR / 'gold_may.parquet'),
                           columns=['n_criticos_60m', 'is_dont_go_next_60m'])
y_val = may_data['is_dont_go_next_60m'].cast(pl.Int8).to_numpy()
x_val = may_data['n_criticos_60m'].to_numpy()

best_f1_val, best_thr = 0, 1
for thr in range(0, 25):
    y_pred = (x_val >= thr).astype(int)
    f1 = f1_score(y_val, y_pred, zero_division=0)
    if f1 > best_f1_val:
        best_f1_val, best_thr = f1, thr
print(f"Regra ótima (val): n_criticos_60m >= {best_thr}  (F1 val = {best_f1_val:.4f})")

# Avaliação no conjunto de teste (jun/2025)
jun_data = pl.read_parquet(str(GOLD_DIR / 'gold_jun.parquet'),
                           columns=['n_criticos_60m', 'is_dont_go_next_60m'])
y_test = jun_data['is_dont_go_next_60m'].cast(pl.Int8).to_numpy()
x_test = jun_data['n_criticos_60m'].to_numpy()

y_pred_base = (x_test >= best_thr).astype(int)
base_f1   = f1_score(y_test, y_pred_base, zero_division=0)
base_prec = precision_score(y_test, y_pred_base, zero_division=0)
base_rec  = recall_score(y_test, y_pred_base, zero_division=0)

lgbm_f1   = final_metrics['f1_score']
lgbm_prec = final_metrics['precision']
lgbm_rec  = final_metrics['recall']

print(f"\n{'Modelo':<25} {'F1':>8} {'Precision':>12} {'Recall':>10}")
print("-" * 58)
print(f"{'Regra (n_criticos>='+str(best_thr)+')':<25} {base_f1:>8.4f} {base_prec:>12.4f} {base_rec:>10.4f}")
print(f"{'LightGBM (final)':<25} {lgbm_f1:>8.4f} {lgbm_prec:>12.4f} {lgbm_rec:>10.4f}")
print(f"\n  Melhora F1       : +{lgbm_f1 - base_f1:.4f}  ({lgbm_f1/base_f1:.1f}x)")
print(f"  Melhora Precision: +{lgbm_prec - base_prec:.4f}  ({lgbm_prec/base_prec:.1f}x)")
print(f"  Falsos Positivos: regra dispara {y_pred_base.sum():,} alertas vs {int(round(35203/lgbm_prec * (1-lgbm_prec))):,} do LightGBM")

In [ ]:
df_comp = pd.DataFrame({
    'Modelo': ['Regra Estática\n(n_criticos_60m≥4)', 'LightGBM\n(54 features)'],
    'F1-Score': [base_f1, lgbm_f1],
    'Precision': [base_prec, lgbm_prec],
    'Recall': [base_rec, lgbm_rec],
})

fig = go.Figure()
cores = ['#EF553B', '#00CC96']
for i, metrica in enumerate(['F1-Score', 'Precision', 'Recall']):
    fig.add_trace(go.Bar(
        name=metrica,
        x=df_comp['Modelo'],
        y=df_comp[metrica],
        text=[f'{v:.3f}' for v in df_comp[metrica]],
        textposition='outside',
    ))

fig.update_layout(
    title='Comparação: Regra Estática vs LightGBM — Conjunto de Teste (Jun/2025)',
    yaxis=dict(title='Score', range=[0, 1.0]),
    barmode='group',
    height=480,
    legend_title='Métrica',
)
fig.show()
fig.write_html(str(DASH_DIR / '07_baseline_comparison.html'))

## 2. Impacto Operacional — O Que os Números Significam na Prática

Com base nos resultados do conjunto de teste (Jun/2025):

In [ ]:
# Dados do conjunto de teste (Jun/2025)
n_total   = 7_854_243   # eventos de telemetria em junho
n_pos     = 35_203      # eventos que precedem Don't Go em ≤60 min
TP        = int(round(n_pos * lgbm_rec))   # DGs antecipados corretamente
FP        = int(round(TP / lgbm_prec * (1 - lgbm_prec)))  # falsos alertas
FN        = n_pos - TP                      # DGs não detectados

# Premissas operacionais (conservadoras)
ANTECEDENCIA_MEDIA_H = 0.75  # 45 min de antecedência efetiva em média
DURACAO_PARADA_H     = 4.0   # parada não planejada típica por DG: 4 horas
CUSTO_H_PARADA       = 50_000  # R$/h de equipamento parado (caminhão 793-D médio)

horas_antecipadas = TP * ANTECEDENCIA_MEDIA_H
paradas_potencialmente_evitaveis = TP  # cada TP = um DG com tempo para agir
horas_impacto = paradas_potencialmente_evitaveis * DURACAO_PARADA_H * 0.30  # 30% conversão em janela de manutenção planejada

print("=" * 60)
print("  IMPACTO OPERACIONAL — JUNHO 2025 (conjunto de teste)")
print("=" * 60)
print(f"  Eventos Don't Go monitorados   : {n_pos:>8,}")
print(f"  DGs antecipados (TP)           : {TP:>8,}  ({lgbm_rec:.1%} dos DGs)")
print(f"  Antecedência média estimada    : {ANTECEDENCIA_MEDIA_H*60:.0f} min antes do DG")
print(f"  Alertas desnecessários (FP)    : {FP:>8,}  ({FP/(FP+TP):.1%} dos alertas emitidos)")
print(f"  DGs não detectados (FN)        : {FN:>8,}  ({lgbm_f1:.1%} F1 geral)")
print()
print(f"  Com ação em 30% dos TPs:")
print(f"  → Horas de parada evitável/mês : ~{horas_impacto:,.0f} h")
print()
print("  INTERPRETAÇÃO PARA OPERAÇÕES:")
print(f"  ✔ Regra estática dispararia {y_pred_base.sum():,} alertas em junho")
print(f"    → {y_pred_base.sum()/(FP+TP):.0f}x mais falsos alarmes que o LightGBM")
print(f"  ✔ LightGBM: 76% dos alertas são reais — operação confia no sistema")
print(f"  ✔ Equipe de manutenção tem ~45 min para reagir a cada alerta")
print("=" * 60)

## 3. Alarm Fingerprint — O DNA do Don't Go

O conceito de **Alarm Fingerprint** é o diferencial mais forte da solução:
em vez de reagir ao Don't Go quando ele ocorre, identificamos quais combinações
de alarmes precedem o evento.

Os 2 alarmes mais preditivos (por importância SHAP e ganho LightGBM):

In [ ]:
import lightgbm as lgb

# Top features do fingerprint por ganho
feat_names = model.feature_name_
gains = model.booster_.feature_importance(importance_type='gain')
df_imp = pd.DataFrame({'feature': feat_names, 'gain': gains}).sort_values('gain', ascending=False)

fp_imp = df_imp[df_imp['feature'].str.startswith('fp_alarm_')].head(10).copy()
fp_imp['alarm_id'] = fp_imp['feature'].str.replace('fp_alarm_', '').astype(int)

# Nomes dos alarmes (via silver)
alarm_names_df = (
    pl.scan_parquet(str(SILVER_DIR / 'silver_jun.parquet'))
    .filter(pl.col('Id_Alarme').is_in(fp_imp['alarm_id'].tolist()))
    .select(['Id_Alarme', 'Alarme', 'Tipo'])
    .unique()
    .collect()
    .to_pandas()
)

fp_imp = fp_imp.merge(alarm_names_df, left_on='alarm_id', right_on='Id_Alarme', how='left')
fp_imp['gain_rel'] = fp_imp['gain'] / fp_imp['gain'].max()

print("Top-10 alarmes mais preditivos (Alarm Fingerprint):")
print(f"{'Rank':<5} {'Id_Alarme':<14} {'Gain rel.':<12} {'Nome do Alarme'}")
print("-" * 80)
for rank, (_, row) in enumerate(fp_imp.iterrows(), 1):
    nome = (row.get('Alarme', 'N/A') or 'N/A')[:55]
    print(f"  {rank:<4} {int(row['alarm_id']):<14} {row['gain_rel']:.3f}         {nome}")

In [ ]:
fp_imp_plot = fp_imp.head(8).copy()
fp_imp_plot['label'] = fp_imp_plot.apply(
    lambda r: f"{(r.get('Alarme','') or '')[:40]}\n(ID {int(r['alarm_id'])})", axis=1
)

fig2 = px.bar(
    fp_imp_plot.sort_values('gain_rel'),
    x='gain_rel', y='label',
    orientation='h',
    title='Alarm Fingerprint — Top-8 Alarmes Preditivos de Don\'t Go (importância por ganho)',
    labels={'gain_rel': 'Importância relativa', 'label': ''},
    color='gain_rel',
    color_continuous_scale='Reds',
)
fig2.update_layout(height=500, showlegend=False, coloraxis_showscale=False,
                   yaxis={'categoryorder': 'total ascending'})
fig2.show()
fig2.write_html(str(DASH_DIR / '08_alarm_fingerprint_narrative.html'))

In [ ]:
# Qual fração dos eventos DG é precedida pelo alarme mais preditivo?
TOP_ALARM_ID = int(fp_imp.iloc[0]['alarm_id'])
TOP_ALARM_NOME = str(fp_imp.iloc[0].get('Alarme', TOP_ALARM_ID))

top_col = f'fp_alarm_{TOP_ALARM_ID}'
jun_fp = pl.read_parquet(str(GOLD_DIR / 'gold_jun.parquet'),
                         columns=[top_col, 'is_dont_go_next_60m'])

n_dg          = jun_fp['is_dont_go_next_60m'].cast(pl.Int8).sum()
n_dg_com_fp   = jun_fp.filter(
    (pl.col('is_dont_go_next_60m').cast(pl.Int8) == 1) & (pl.col(top_col) == 1)
).height
n_fp_total    = jun_fp.filter(pl.col(top_col) == 1).height
n_fp_sem_dg   = n_fp_total - n_dg_com_fp

print(f"Alarme mais preditivo: '{TOP_ALARM_NOME[:60]}' (ID {TOP_ALARM_ID})")
print()
print(f"  Eventos DG em junho                    : {n_dg:,}")
print(f"  DG precedidos por este alarme (≤60 min): {n_dg_com_fp:,}  ({n_dg_com_fp/n_dg:.1%})")
print(f"  Ocorrências do alarme sem DG            : {n_fp_sem_dg:,}")
print()
print(f"  ► {n_dg_com_fp/n_dg:.0%} dos Don't Go ocorrem após este alarme estar ativo")
print(f"    → Presença deste alarme sozinho já é um sinal de risco elevado")

## 4. Recomendações Operacionais

Com base nos padrões identificados pela análise exploratória (notebooks 01–03),
validação das hipóteses de negócio e resultados do modelo preditivo:

In [ ]:
recomendacoes = [
    {
        "numero": 1,
        "titulo": "Protocolo de Inspeção Preventiva ao Detectar Alarm Fingerprint",
        "descricao": (
            "Quando o modelo emitir alerta (P(DG) ≥ 0.93), acionar protocolo de "
            "inspeção em até 30 minutos. O alarme 'Raise Hoist Limited By End Of "
            "Stroke' (ID 1074008260) isoladamente cobre mais de 60% dos Don't Go "
            "em caminhões — integrá-lo ao checklist diário reduz tempo de reação."
        ),
        "hipotese_validada": "H1 (sequência característica precede DG em até 4h)",
        "impacto": "Redução de paradas não planejadas por antecipação da manutenção",
    },
    {
        "numero": 2,
        "titulo": "Monitoramento Diferenciado por Modelo de Frota",
        "descricao": (
            "Caminhões 793-D e Escavadeiras LeTourneau L 1850 têm perfis de "
            "alarme distintos (H3 validada). A frota é o 5º fator mais preditivo "
            "no SHAP. Recomenda-se criar limiares e rotinas de inspeção "
            "separadas por modelo, em vez de um protocolo genérico."
        ),
        "hipotese_validada": "H3 (modelos de frota distintos têm perfis de falha diferentes)",
        "impacto": "Aumento da precisão dos alertas por segmento de equipamento",
    },
    {
        "numero": 3,
        "titulo": "Janela de Alta Atenção: Últimas 4 Horas de Turno",
        "descricao": (
            "A posição dentro do turno (posicao_turno) e a hora do dia (hora_dia) "
            "são respectivamente o 6º e 3º fatores mais importantes. A incidência "
            "de DG aumenta nas últimas 4 horas de turno, sugerindo fadiga de "
            "componentes por uso acumulado. Inspeções rápidas no intervalo de "
            "meio-turno podem reduzir esse padrão."
        ),
        "hipotese_validada": "H2 (frequência de alarmes críticos aumenta progressivamente)",
        "impacto": "Prevenção de DG no período de maior risco do turno",
    },
]

for r in recomendacoes:
    print(f"\n{'='*65}")
    print(f"  [{r['numero']}] {r['titulo']}")
    print(f"{'='*65}")
    print(f"  {r['descricao']}")
    print(f"\n  Base científica: {r['hipotese_validada']}")
    print(f"  Impacto esperado: {r['impacto']}")

## Resumo Executivo

| Dimensão | Resultado |
|----------|-----------|
| Modelo | LightGBM com 54 features, validação temporal estrita |
| F1-Score (teste jun/2025) | **0.6886** (6.3× superior à regra estática) |
| Precisão | 76% — 3 em cada 4 alertas são reais |
| Cobertura | 63% dos Don't Go detectados com ≥1h de antecedência |
| ROC-AUC | 0.9923 — discriminação quase perfeita |
| Superioridade vs baseline | Precisão 12× maior, F1 6× maior |
| Principal alarme preditivo | Raise Hoist Limited (ID 1074008260) |
| Recomendação prioritária | Protocolo de inspeção em 30 min após alerta |

> **O sistema não substitui a engenharia de manutenção — ele dá à equipe
> o tempo necessário para agir antes que o equipamento seja proibido de operar.**